In [1]:
import openpyxl
import pandas as pd
from openpyxl.utils import get_column_letter

# Load the workbook
wb = openpyxl.load_workbook('PRISMA_Deduplicated.xlsx')
ws = wb['Duplicate Review']

# Read data into a list of dictionaries, SKIP EMPTY ROWS
headers = []
for col in range(1, ws.max_column + 1):
    headers.append(ws.cell(1, col).value)

data = []
for row in range(2, ws.max_row + 1):
    record = {}
    is_empty = True
    for col_idx, header in enumerate(headers, start=1):
        value = ws.cell(row, col_idx).value
        record[header] = value
        if value is not None:
            is_empty = False
    
    if not is_empty:
        data.append(record)

print(f"Total records before title deduplication: {len(data)}")

# Convert to DataFrame
df = pd.DataFrame(data)

# Count records with and without titles
records_with_title = df['Title'].notna().sum()
records_without_title = df['Title'].isna().sum()
print(f"Records with Title: {records_with_title}")
print(f"Records without Title: {records_without_title}")

# Separate records with and without Title
df_with_title = df[df['Title'].notna()].copy()
df_without_title = df[df['Title'].isna()].copy()

# Remove duplicates based on Title (keep first occurrence)
df_with_title_dedup = df_with_title.drop_duplicates(subset=['Title'], keep='first')

duplicates_removed = len(df_with_title) - len(df_with_title_dedup)
print(f"Duplicates removed based on Title: {duplicates_removed}")

# Combine back
df_final = pd.concat([df_with_title_dedup, df_without_title], ignore_index=True)

print(f"Total records after title deduplication: {len(df_final)}")

# Create new sheet for deduplicated records
sheet_name = 'Duplicate Review(DOI + Title)'
if sheet_name in wb.sheetnames:
    del wb[sheet_name]
ws_new = wb.create_sheet(sheet_name)

# Write headers
for col_idx, header in enumerate(headers, start=1):
    ws_new.cell(1, col_idx, header)
    ws_new.cell(1, col_idx).font = openpyxl.styles.Font(bold=True)

# Write deduplicated data
for row_idx, record in enumerate(df_final.to_dict('records'), start=2):
    for col_idx, header in enumerate(headers, start=1):
        ws_new.cell(row_idx, col_idx, record[header])

# Auto-adjust column widths
for col_idx, header in enumerate(headers, start=1):
    column_letter = get_column_letter(col_idx)
    if header == 'Title':
        ws_new.column_dimensions[column_letter].width = 50
    elif header == 'Author/(s)':
        ws_new.column_dimensions[column_letter].width = 30
    elif header == 'Abstract':
        ws_new.column_dimensions[column_letter].width = 60
    elif header == 'Keywords':
        ws_new.column_dimensions[column_letter].width = 30
    elif header == 'Year':
        ws_new.column_dimensions[column_letter].width = 10
    elif header == 'Platform':
        ws_new.column_dimensions[column_letter].width = 20
    elif header == 'DOI':
        ws_new.column_dimensions[column_letter].width = 25

# Save the workbook
wb.save('PRISMA_Deduplicated.xlsx')

print(f"\nTitle deduplication complete!")
print(f"Deduplicated records ({len(df_final)}) saved to '{sheet_name}' sheet")

Total records before title deduplication: 6350
Records with Title: 6350
Records without Title: 0
Duplicates removed based on Title: 71
Total records after title deduplication: 6279

Title deduplication complete!
Deduplicated records (6279) saved to 'Duplicate Review(DOI + Title)' sheet
